<a href="https://colab.research.google.com/github/2303A51190/RL-Lab/blob/main/RL_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# ASSIGNMENT 1: BELLMAN OPTIMALITY AND VALUE ITERATION
# Deterministic 2 x 2 Grid World
# ============================================================

# 1. Define the MDP
states = ["S0", "S1"]
actions = ["Up", "Down", "Left", "Right"]
gamma = 0.9
theta = 1e-6

goal = "G"
wall = "Wall"

# Rewards
normal_reward = -1
wall_reward = -1
goal_reward = 10

# Grid:
# S0 | Wall
# S1 | G

# Deterministic transitions:
# Invalid moves and moves into the wall leave the agent
# in its current state and give reward -1.

transitions = {
    ("S0", "Up"):    ("S0", wall_reward),
    ("S0", "Down"):  ("S1", normal_reward),
    ("S0", "Left"):  ("S0", wall_reward),
    ("S0", "Right"): ("S0", wall_reward),

    ("S1", "Up"):    ("S0", normal_reward),
    ("S1", "Down"):  ("S1", wall_reward),
    ("S1", "Left"):  ("S1", wall_reward),
    ("S1", "Right"): (goal, goal_reward)
}

# 2. Inspect the MDP
print("=" * 65)
print("1. MDP INFORMATION")
print("=" * 65)
print("States:", states)
print("Actions:", actions)
print("Terminal state:", goal)
print("Wall location:", wall)
print("Discount factor (gamma):", gamma)
print("Normal movement reward:", normal_reward)
print("Wall/boundary reward:", wall_reward)
print("Goal reward:", goal_reward)

print("\n2. TRANSITION VERIFICATION")
print("-" * 65)
for (state, action), (next_state, reward) in transitions.items():
    print(f"State: {state:2} | Action: {action:5} | "
          f"Next state: {next_state:4} | Reward: {reward}")

# Verify that every state-action pair has a transition
assert len(transitions) == len(states) * len(actions)
print("\nAll state-action pairs have defined transitions.")

# 3. Initialize state values
V = {"S0": 0.0, "S1": 0.0, "G": 0.0}

print("\n3. INITIAL STATE VALUES")
print(V)

# 4 and 5. Synchronous Value Iteration and convergence
print("\n4. VALUE ITERATION")
print("-" * 65)
print(f"{'Iteration':<12}{'V(S0)':>12}{'V(S1)':>12}{'Delta':>14}")

iteration = 0
while True:
    iteration += 1
    old_V = V.copy()
    new_V = old_V.copy()

    # Synchronous update: use old_V for every calculation
    for state in states:
        q_values = []
        for action in actions:
            next_state, reward = transitions[(state, action)]
            q = reward + gamma * old_V[next_state]
            q_values.append(q)

        new_V[state] = max(q_values)

    # Goal is terminal
    new_V["G"] = 0.0

    delta = max(abs(new_V[s] - old_V[s]) for s in states)
    V = new_V

    print(f"{iteration:<12}{V['S0']:>12.6f}"
          f"{V['S1']:>12.6f}{delta:>14.6f}")

    if delta < theta:
        break

print("\n6. FINAL OPTIMAL STATE VALUES")
for state in ["S0", "S1", "G"]:
    print(f"V*({state}) = {V[state]:.6f}")

# 6 and 7. Calculate Q-values and extract optimal policy
print("\n7. ACTION VALUES AND OPTIMAL POLICY")
print("-" * 65)

Q = {}
policy = {}

for state in states:
    Q[state] = {}

    for action in actions:
        next_state, reward = transitions[(state, action)]
        Q[state][action] = reward + gamma * V[next_state]

    best_value = max(Q[state].values())
    best_actions = [
        action for action, value in Q[state].items()
        if abs(value - best_value) < 1e-9
    ]
    policy[state] = best_actions

    print(f"\nState: {state}")
    for action in actions:
        print(f"Q({state}, {action}) = {Q[state][action]:.6f}")

    print(f"Optimal action(s): {', '.join(best_actions)}")

# 8. Verify Bellman Optimality
print("\n8. BELLMAN OPTIMALITY VERIFICATION")
print("-" * 65)

for state in states:
    max_q = max(Q[state].values())
    print(f"State {state}: V*({state}) = {V[state]:.6f}, "
          f"max Q*({state}, a) = {max_q:.6f}")

    assert abs(V[state] - max_q) < 1e-6

print("\nVerification successful: V*(s) = max_a Q*(s, a).")

